# Lithium supply chain optimisation - Energies 2024

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/lithium-optsc-energies-2024/blob/main/notebooks/00_walkthrough.ipynb)

The model behind [Jones (2024), *Energies* 17, 2685](https://doi.org/10.3390/en17112685).

**This notebook is thin on purpose.** It imports the package and calls it; it contains no model
logic, so it cannot drift from the code that produced the paper. To read the model, read
`src/lithium_energies/model.py`.


## 1. Install

On Colab, install the package from the repository. Locally, `pip install -e ".[dev]"` once.

In [1]:
# Colab only. Locally you have already run `pip install -e ".[dev]"`.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/sear-labs/lithium-optsc-energies-2024.git"], check=True)
else:
    print("local environment - skipping install")

local environment - skipping install


## 2. Gurobi licence

The model is far larger than the free `pip` licence allows.

A node-locked licence file cannot work in Colab - the VM is a different machine every session - so
use WLS credentials held as **Colab secrets** (key icon, left sidebar): `GRB_WLSACCESSID`,
`GRB_WLSSECRET`, `GRB_LICENSEID`.

`SecretNotFoundError` here is the expected first run for anyone who has not added them.

In [2]:
import gurobipy as gp

_HELP = [
    "Gurobi licence not available.",
    "In Colab, add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as secrets",
    "(key icon in the left sidebar) from your Gurobi WLS account.",
]

try:
    from google.colab import userdata

    options = {
        "WLSACCESSID": userdata.get("GRB_WLSACCESSID"),
        "WLSSECRET": userdata.get("GRB_WLSSECRET"),
        # userdata.get always returns a str; Gurobi needs an int here.
        "LICENSEID": int(userdata.get("GRB_LICENSEID")),
    }
    env = gp.Env(params=options)
    print("using the WLS licence from Colab secrets")
except ImportError:
    # Not on Colab: take whatever licence the machine has.
    env = gp.Env()
    print("local environment - using the machine licence")
except Exception as exc:
    raise SystemExit(" ".join(_HELP) + f" Original error: {exc}")


Set parameter Username


Set parameter LicenseID to value 2750151


Academic license - for non-commercial use only - expires 2026-12-04


local environment - using the machine licence


## 3. Build the instance and check it against the paper

The paper states the model had **13,556 rows** and **10,706 continuous variables**. Rebuilding from
`data/raw/` should reproduce both exactly - that is the check that the data pipeline still produces
the published instance.

In [3]:
import subprocess, sys
print(subprocess.run([sys.executable, "../scripts/run_all.py"],
                     capture_output=True, text=True).stdout)

building the model from data/input_csvs ...
['spod', 'clay', 'brine', 'lce', 'loh', 'cath_nmc', 'cath_lfp', 'GWh_nmc', 'GWh_lfp', 'bev_nmc', 'phev_nmc', 'bev_lfp', 'phev_lfp']
Set parameter Username
Set parameter LicenseID to value 2750151
Academic license - for non-commercial use only - expires 2026-12-04
Set parameter TimeLimit to value 3600
Read solution from file C:\Users\jonesec\OneDrive - UT Arlington\Documents\Inventory\projects\lithium-energies-2024\data\raw\warm_start.sol
[post-build cell failed, expected: AttributeError: Unable to retrieve attribute 'ObjVal']

                                rebuilt        paper   match
------------------------------------------------------------------
constraint rows                   13556        13556   YES
continuous variables              10706        10706   YES
integer/binary variables           2673            -   

build time: 7.7 s

DIMENSIONS REPRODUCE THE PAPER



## 4. Solve

Warm-started from `data/raw/warm_start.sol`, which is an **input**: the paper's run was
itself warm-started, cumulatively, to reach 28 hours of solve time.

600 s gets within ~3 units of the published 9,511,432. It will not match exactly - the model stops
on a time limit with a ~0.03% gap still open, so the published number is an incumbent, not an
optimum. See the README.

In [4]:
print(subprocess.run([sys.executable, "../scripts/run_all.py", "--solve", "600"],
                     capture_output=True, text=True).stdout)

building the model from data/input_csvs ...
['spod', 'clay', 'brine', 'lce', 'loh', 'cath_nmc', 'cath_lfp', 'GWh_nmc', 'GWh_lfp', 'bev_nmc', 'phev_nmc', 'bev_lfp', 'phev_lfp']
Set parameter Username
Set parameter LicenseID to value 2750151
Academic license - for non-commercial use only - expires 2026-12-04
Set parameter TimeLimit to value 3600
Read solution from file C:\Users\jonesec\OneDrive - UT Arlington\Documents\Inventory\projects\lithium-energies-2024\data\raw\warm_start.sol
Set parameter TimeLimit to value 600
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i9-10900 CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 13556 rows, 13379 columns and 53555 nonzeros (Min)
Model fingerprint: 0x2040d7c5
Model has 1 linear objective coefficients
Variable types: 10706 continuous, 2673 inte

## 5. What to cite

Cite **the paper** for the work and **this repository** for the code. `CITATION.cff` in the repo
root carries both, and GitHub renders a "Cite this repository" button from it.